In [0]:
raw_table_path = "/Volumes/workspace/default/metricmind_raw/delta/dataco"

df_raw = spark.read.format("delta").load(raw_table_path)

print("Raw Delta loaded successfully")

In [0]:
df_raw.printSchema()

In [0]:
raw_count = df_raw.count()
print("Raw row count:", raw_count)

In [0]:
df_staging = df_raw

print("Fresh staging DataFrame created from raw data")

In [0]:
import re

column_mapping = {
    col_name: re.sub(
        r"_+",
        "_",
        re.sub(r"[^a-zA-Z0-9]+", "_", col_name).strip("_").lower()
    )
    for col_name in df_staging.columns
}

df_staging = df_staging.select([
    df_staging[old_name].alias(new_name)
    for old_name, new_name in column_mapping.items()
])

print("Column names standardized")

In [0]:
from pyspark.sql.functions import to_date, substring_index, col

df_staging = (
    df_staging
    .withColumn(
        "order_date_dateorders",
        to_date(
            substring_index(col("order_date_dateorders"), " ", 1),
            "M/d/yyyy"
        )
    )
    .withColumn(
        "shipping_date_dateorders",
        to_date(
            substring_index(col("shipping_date_dateorders"), " ", 1),
            "M/d/yyyy"
        )
    )
)

print("Date columns converted successfully")

In [0]:
df_staging.select(
    "order_date_dateorders",
    "shipping_date_dateorders"
).show(5, truncate=False)

In [0]:
from pyspark.sql.functions import col,sum as spark_sum

null_counts = df_staging.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_staging.columns
])

null_counts.show(truncate=False)

In [0]:
null_profile = []

for c in df_staging.columns:
    count = df_staging.filter(col(c).isNull()).count()
    if count > 0:
        null_profile.append((c, count))

for c, count in null_profile:
    print(f"{c}: {count}")
    

In [0]:
total_rows = df_staging.count()
distinct_rows = df_staging.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate rows:", total_rows - distinct_rows)

In [0]:
from pyspark.sql.functions import year, month, quarter

df_staging = (
    df_staging
    .withColumn("order_year", year("order_date_dateorders"))
    .withColumn("order_month", month("order_date_dateorders"))
    .withColumn("order_quarter", quarter("order_date_dateorders"))
)

print("Time fields created")

In [0]:
from pyspark.sql.functions import trim, upper

df_staging = df_staging.withColumn(
    "region",
    upper(trim("order_region"))
)

print("Region field standardized")

In [0]:
df_staging.select(
    "order_region",
    "region",
    "order_date_dateorders",
    "order_year",
    "order_month",
    "order_quarter"
).show(10, truncate=False)

In [0]:
staging_path = "/Volumes/workspace/default/metricmind_raw/delta/staging_dataco"
(
    df_staging.write
    .format("delta")
    .mode("overwrite")
    .save(staging_path)
)

print("Staging Delta table written sucessfully")
print("Path:", staging_path)

In [0]:
df_staging_saved = (
    spark.read
    .format("delta")
    .load(staging_path)
)

print("Staging Delta loaded successfully")
print("Rows:", df_staging_saved.count())

In [0]:
df_staging_saved.select(
    "order_date_dateorders",
    "shipping_date_dateorders",
    "region",
    "order_year",
    "order_month",
    "order_quarter"
).show(5, truncate=False)

In [0]:
raw_count = df_raw.count()
staging_count = df_staging_saved.count()

print("Raw row count: ", raw_count)
print("Staging row count: ", staging_count)
print("Difference: ", raw_count - staging_count)

In [0]:
df_staging_saved.select(
    "order_date_dateorders",
    "shipping_date_dateorders",
    "region",
    "order_year",
    "order_month",
    "order_quarter"
).printSchema()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

quality_check = df_staging_saved.select(
    spark_sum(col("order_id").isNull().cast("int")).alias("null_order_id"),
    spark_sum(col("order_date_dateorders").isNull().cast("int")).alias("null_order_date"),
    spark_sum(col("region").isNull().cast("int")).alias("null_region")
)
quality_check.show()

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

In [0]:
print("Staging DataFrame:")
print(df_staging)

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.stg_dataco
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/workspace/default/metricmind_raw/delta/staging_dataco`
""")

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

In [0]:
df_staging.select("order_item_id").count()

In [0]:
df_staging.select("order_item_id").distinct().count()